In [ ]:
# STEP 1: Install the Agents SDK (run this cell first)
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic

In [ ]:
# STEP 2: Connect to Groq (free key from console.groq.com/keys)
import os
from openai import OpenAI, AsyncOpenAI
from IPython.display import display, Markdown
from google.colab import userdata                 # Colab's secret store
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

groq_api_key = userdata.get("GROQ_API_KEY")       # secret name in Colab: GROQ_API_KEY
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq's OpenAI-compatible address

# Same OpenAI library, different address: requests now go to Groq, not OpenAI
groq_client = AsyncOpenAI(api_key = groq_api_key, base_url = GROQ_BASE_URL)

# Traces are uploaded to OpenAI's dashboard and need an OpenAI key, so switch them off
set_tracing_disabled(True)

print("Groq client successfully configured.")
print(groq_api_key[:5])                           # should print: gsk_x

In [ ]:
# STEP 3: Choose the models and see which ones your key can use
GROQ_MODEL     = "openai/gpt-oss-20b"     # small and very fast  (replaces gpt-4.1-mini)
GROQ_BIG_MODEL = "openai/gpt-oss-120b"    # bigger and smarter   (replaces the larger GPT models)

def groq(model_name):
    """Wraps a Groq model name so the Agents SDK sends it to Groq exactly as written."""
    return OpenAIChatCompletionsModel(model = model_name, openai_client = groq_client)

# Model names change over time. If you get "404 model not found", pick a name from this list.
sync_client = OpenAI(api_key = groq_api_key, base_url = GROQ_BASE_URL)
for m in sync_client.models.list().data:
    print(m.id)

In [ ]:
from agents import Agent

fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
3. if the input is not a fact checking question and its a general question donot use any verdict prefix, give just a plane response
"""

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",                    # badge name
                           instructions = fact_checker_instructions, # job description
                           model = groq(GROQ_MODEL))                 # which brain (now on Groq)

print(f"Agent '{fact_checker_agent.name}' created successfully!")

In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [ ]:
from agents import Runner

statement = "The Great Wall of China is visible from space with the naked eye."
print_markdown(f"Asking the Fact Checker to verify: '{statement}'")

response = await Runner.run(
    starting_agent = fact_checker_agent,   # WHICH agent to run
    input = statement                      # WHAT to give it
)

print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)      # the agent's final answer

In [ ]:
# Traces are off, so check token usage here instead of the OpenAI dashboard
usage = response.context_wrapper.usage
print(f"Requests: {usage.requests} | Input tokens: {usage.input_tokens} | "
      f"Output tokens: {usage.output_tokens} | Total: {usage.total_tokens}")

In [ ]:
check_recall_statement = "What did we discuss in the last message?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)

In [ ]:
check_recall_statement = "Do you remmember the fact check statement/question?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = groq(GROQ_BIG_MODEL))

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)

In [ ]:
from agents import ModelSettings
from openai.types.shared import Reasoning

# Same small model, but told to "think less": faster, shorter answers
fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = groq(GROQ_MODEL),
                           model_settings = ModelSettings(reasoning = Reasoning(effort = "low")))

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)

In [ ]:
tweet_bot_instructions = """
Instruction:
Write a short, engaging tweet (≤280 characters) about the given topic. The tweet must
include at least one relevant emoji and one relevant hashtag.

Context:
You are **TweetBot**, a witty and creative social-media expert known for crafting viral tweets.

Input:
A topic provided by the user.

Output:
A single tweet (≤280 characters) about the topic, containing at least one relevant emoji
and one relevant hashtag.
"""

tweet_bot_agent = Agent(name = "Tweet Bot",
                        instructions = tweet_bot_instructions,
                        model = groq(GROQ_BIG_MODEL))

topic = "The future of renewable energy"
tweet_response = await Runner.run(starting_agent = tweet_bot_agent, input = topic)
print_markdown(tweet_response.final_output)